# Olist E-Commerce Data Wrangling
## Task 1 – Data Immersion & Cleaning

This notebook demonstrates:

- Data loading and inspection  
- Datetime conversion  
- Missing value analysis  
- Feature engineering  
- Multi-table merging  
- Revenue metric creation  

The goal is to prepare an analysis-ready master dataset.

## 1️⃣ Import Required Libraries

We begin by importing the necessary Python libraries for data manipulation and analysis.

In [1]:
import pandas as pd

## 2️⃣ Load Orders Dataset

The orders dataset contains transactional-level information including:

- Order ID
- Customer ID
- Order status
- Purchase timestamp
- Delivery timestamps

We begin by loading and inspecting the structure of the dataset.

In [2]:
order=pd.read_csv("olist_orders_dataset.csv")
order.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 3️⃣ Initial Data Inspection

We inspect the dataset structure to understand:

- Number of rows and columns  
- Data types of each column  
- Presence of missing values  

This helps identify cleaning requirements before transformation.

In [3]:
order.shape

(99441, 8)

In [4]:
order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## 4️⃣ Datetime Conversion

The delivery and purchase timestamps were originally stored as object (string) type.

To enable:

- Time-based analysis  
- Delivery duration calculation  
- Feature engineering  

We convert all relevant timestamp columns to proper `datetime64` format.

In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    order[col] = pd.to_datetime(order[col])

In [6]:
order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


## 5️⃣ Missing Value Analysis

We examine the dataset for missing values to understand:

- Which columns contain null values  
- Whether missing data is random or business-driven  

Observation:
Missing delivery timestamps typically correspond to non-delivered or cancelled orders, indicating logical business behavior rather than data corruption.

In [7]:
order.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [8]:
order['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
order.duplicated().sum()

np.int64(0)

## 6️⃣ Feature Engineering – Time-Based Attributes

To enable time-series and trend analysis, we extract additional attributes from the purchase timestamp:

- Purchase year  
- Purchase month  
- Purchase day  
- Purchase weekday  

These features support:
- Monthly sales trend analysis  
- Weekday purchase behavior analysis  
- Seasonal pattern identification  

In [10]:
order['purchase_year'] = order['order_purchase_timestamp'].dt.year
order['purchase_month'] = order['order_purchase_timestamp'].dt.month
order['purchase_day'] = order['order_purchase_timestamp'].dt.day
order['purchase_weekday'] = order['order_purchase_timestamp'].dt.day_name()

In [11]:
order[['order_purchase_timestamp','purchase_year','purchase_month','purchase_weekday']].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_weekday
0,2017-10-02 10:56:33,2017,10,Monday
1,2018-07-24 20:41:37,2018,7,Tuesday
2,2018-08-08 08:38:49,2018,8,Wednesday
3,2017-11-18 19:28:06,2017,11,Saturday
4,2018-02-13 21:18:39,2018,2,Tuesday


In [12]:
order['delivery_days'] = (
    order['order_delivered_customer_date'] - 
    order['order_purchase_timestamp']
).dt.days

In [13]:
order[['order_status','delivery_days']].head(10)

,order_status,delivery_days
0,delivered,8.0
1,delivered,13.0
2,delivered,9.0
3,delivered,13.0
4,delivered,2.0
5,delivered,16.0
6,invoiced,NaN
7,delivered,9.0
8,delivered,9.0
9,delivered,18.0


## 7️⃣ Delivery Performance Metrics

To evaluate logistics efficiency, we engineer the following KPIs:

- **delivery_days** → Actual time taken to deliver an order  
- **estimated_delivery_days** → Expected delivery duration  
- **delivery_delay** → Difference between actual and estimated delivery  

Interpretation:
- Negative delay → Delivered earlier than estimated  
- Positive delay → Delivered late  
- Null values → Orders not delivered  

These metrics enable logistics performance and service-level analysis.

In [14]:
order['estimated_delivery_days'] = (
    order['order_estimated_delivery_date'] -
    order['order_purchase_timestamp']
).dt.days

order['delivery_delay'] = order['delivery_days'] - order['estimated_delivery_days']

In [15]:
order[['delivery_days','estimated_delivery_days','delivery_delay']].head(10)

,delivery_days,estimated_delivery_days,delivery_delay
0,8.0,15,-7.0
1,13.0,19,-6.0
2,9.0,26,-17.0
3,13.0,26,-13.0
4,2.0,12,-10.0
5,16.0,22,-6.0
6,NaN,27,NaN
7,9.0,21,-12.0
8,9.0,41,-32.0
9,18.0,24,-6.0


In [16]:
order.to_csv("cleaned_orders_dataset.csv", index=False)

## 8️⃣ Load Order Items Dataset

The order_items table contains item-level transactional data, including:

- Product ID  
- Seller ID  
- Price  
- Freight value  

This table enables revenue-level and product-level analysis.

In [17]:
order_items = pd.read_csv("olist_order_items_dataset.csv")

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [18]:
order_items.shape

(112650, 7)

In [19]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


## 9️⃣ Multi-Table Integration

We merge the cleaned orders dataset with the order_items table using `order_id`.

Why?

- Orders table → Order-level information  
- Order_items table → Product-level revenue information  

The merge enables:

- Revenue calculation  
- Item-level analysis  
- Customer purchase behavior analysis  

In [20]:
master_df = order.merge(order_items, on="order_id", how="left")

master_df.shape

(113425, 21)

## 🔟 Revenue Metric Creation

To prepare the dataset for financial and sales analysis, we engineer:

**total_revenue = product price + freight value**

This metric represents total customer payment per item and serves as the foundation for:

- Revenue reporting  
- Sales trend analysis  
- Profitability exploration  

In [21]:
master_df['total_revenue'] = master_df['price'] + master_df['freight_value']

master_df[['price','freight_value','total_revenue']].head()

,price,freight_value,total_revenue
0,29.99,8.72,38.71
1,118.70,22.76,141.46
2,159.90,19.22,179.12
3,45.00,27.20,72.20
4,19.90,8.72,28.62


## 1️⃣1️⃣ Final Data Validation

After merging and feature engineering, we perform a final validation step:

- Check for unexpected missing values  
- Identify incomplete merged records  
- Ensure dataset consistency  

Minor inconsistent rows (<1%) were removed to maintain analytical integrity.

In [22]:
master_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_estimated_delivery_date       0
purchase_year                       0
purchase_month                      0
purchase_day                        0
purchase_weekday                    0
delivery_days                    3229
estimated_delivery_days             0
delivery_delay                   3229
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
total_revenue                     775
dtype: int64

In [23]:
master_df.to_csv("cleaned_master_olist_dataset.csv", index=False)

In [24]:
master_df = master_df.dropna(subset=['order_item_id'])

master_df.shape

(112650, 22)

## 1️⃣2️⃣ Export Clean Master Dataset

The final cleaned and feature-engineered dataset is exported as:

`cleaned_master_olist_dataset.csv`

This dataset is now:

- Analysis-ready  
- Suitable for EDA  
- Ready for KPI development  
- Compatible with BI tools (Power BI / Tableau)  

In [25]:
master_df.to_csv("cleaned_master_olist_dataset.csv", index=False)

In [26]:
sample_df = master_df.sample(5000, random_state=42)
sample_df.to_csv("cleaned_master_sample.csv", index=False)

sample_df.shape

(5000, 22)